<a href="https://colab.research.google.com/github/ajaykumar080286/MachineLearning/blob/main/68_Optuna_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.3 MB/s eta 0:00:00


In [3]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [7]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-06-28 07:31:40,135] A new study created in memory with name: no-name-0995af0e-5dd5-43ba-a0b8-598878dbd6e8
[I 2026-06-28 07:31:42,301] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 152, 'max_depth': 20}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-28 07:31:43,117] Trial 1 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 77, 'max_depth': 4}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-28 07:31:44,072] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 90, 'max_depth': 8}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-28 07:31:45,676] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 147, 'max_depth': 13}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-28 07:31:47,356] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 136, 'max_depth': 18}. Best is trial 4 with value: 0.77281191

In [8]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 54, 'max_depth': 7}


In [9]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Samplers in Optuna

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-06-28 07:34:02,456] A new study created in memory with name: no-name-ba864fa1-06cf-49d4-bb92-1998d5b7e1c2
[I 2026-06-28 07:34:03,709] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 113, 'max_depth': 13}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:34:05,694] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 189, 'max_depth': 14}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:34:06,603] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 84, 'max_depth': 16}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:34:08,823] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 180, 'max_depth': 4}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:34:11,193] Trial 4 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 195, 'max_depth': 18}. Best is trial 4 with value: 0.774674

In [12]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 119, 'max_depth': 18}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [14]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [15]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-06-28 07:36:54,649] A new study created in memory with name: no-name-2bee9589-0540-44b8-9b82-52138464fef2
[I 2026-06-28 07:36:55,514] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:36:57,511] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-28 07:36:58,527] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-28 07:37:00,192] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-28 07:37:01,518] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [16]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [17]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [18]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [19]:
# 1. Optimization History
plot_optimization_history(study).show()

In [20]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [21]:
# 3. Slice Plot
plot_slice(study).show()

In [22]:
# 4. Contour Plot
plot_contour(study).show()

In [23]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

Optimizing Multiple ML Models

In [24]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [25]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [26]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-28 08:46:42,656] A new study created in memory with name: no-name-74514eaa-1ee6-4ac2-90bf-56ba623c54b8
[I 2026-06-28 08:46:42,745] Trial 0 finished with value: 0.6871508379888268 and parameters: {'classifier': 'SVM', 'C': 25.672770959982095, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.6871508379888268.
[I 2026-06-28 08:46:48,822] Trial 1 finished with value: 0.7541899441340781 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 158, 'learning_rate': 0.010340638011415059, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.7541899441340781.
[I 2026-06-28 08:46:49,497] Trial 2 finished with value: 0.756052141527002 and parameters: {'classifier': 'RandomForest', 'n_estimators': 58, 'max_depth': 18, 'min_samples_split': 10, 'min_samples_leaf': 2, 'bootstrap': True}. Best is trial 2 with value: 0.756052141527002.
[I 2026-06-28 08:46:51,921] Trial 3 finished with value: 0.750465549348231 and parame

In [27]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12974262430367706, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [28]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.687151,2026-06-28 08:46:42.663477,2026-06-28 08:46:42.745771,0 days 00:00:00.082294,25.672771,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.754190,2026-06-28 08:46:42.747096,2026-06-28 08:46:48.821987,0 days 00:00:06.074891,NaN,NaN,GradientBoosting,NaN,NaN,0.010341,14.0,4.0,2.0,158.0,COMPLETE
2,2,0.756052,2026-06-28 08:46:48.823947,2026-06-28 08:46:49.497490,0 days 00:00:00.673543,NaN,True,RandomForest,NaN,NaN,NaN,18.0,2.0,10.0,58.0,COMPLETE
3,3,0.750466,2026-06-28 08:46:49.499424,2026-06-28 08:46:51.921035,0 days 00:00:02.421611,NaN,NaN,GradientBoosting,NaN,NaN,0.178412,19.0,2.0,7.0,72.0,COMPLETE
4,4,0.770950,2026-06-28 08:46:51.922433,2026-06-28 08:46:54.127320,0 days 00:00:02.204887,NaN,False,RandomForest,NaN,NaN,NaN,20.0,7.0,8.0,284.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.769088,2026-06-28 08:47:42.530108,2026-06-28 08:47:42.609724,0 days 00:00:00.079616,0.121083,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.787709,2026-06-28 08:47:42.612657,2026-06-28 08:47:42.672314,0 days 00:00:00.059657,0.185303,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.711359,2026-06-28 08:47:42.676436,2026-06-28 08:47:42.731134,0 days 00:00:00.054698,0.148225,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.767225,2026-06-28 08:47:42.735815,2026-06-28 08:47:42.800283,0 days 00:00:00.064468,0.353747,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [29]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,75
RandomForest,15
GradientBoosting,10


In [30]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.748231
RandomForest,0.764246
SVM,0.775295


In [31]:
# 1. Optimization History
plot_optimization_history(study).show()

In [32]:
# 3. Slice Plot
plot_slice(study).show()

In [33]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [36]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

!pip install optuna-integration[xgboost]

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 2.3 MB/s eta 0:00:00


[I 2026-06-28 08:49:15,180] A new study created in memory with name: no-name-0f86fbf7-60fc-4923-bbb9-1a7307f496e2


[0]	train-mlogloss:0.95547	eval-mlogloss:0.94971
[1]	train-mlogloss:0.79783	eval-mlogloss:0.78341
[2]	train-mlogloss:0.67231	eval-mlogloss:0.65247
[3]	train-mlogloss:0.57076	eval-mlogloss:0.54607
[4]	train-mlogloss:0.49039	eval-mlogloss:0.46135
[5]	train-mlogloss:0.42139	eval-mlogloss:0.38795
[6]	train-mlogloss:0.36711	eval-mlogloss:0.33293
[7]	train-mlogloss:0.32231	eval-mlogloss:0.28730
[8]	train-mlogloss:0.28560	eval-mlogloss:0.24725
[9]	train-mlogloss:0.25421	eval-mlogloss:0.21380
[10]	train-mlogloss:0.22741	eval-mlogloss:0.18724
[11]	train-mlogloss:0.20498	eval-mlogloss:0.16141
[12]	train-mlogloss:0.19386	eval-mlogloss:0.15294
[13]	train-mlogloss:0.17556	eval-mlogloss:0.13304
[14]	train-mlogloss:0.16646	eval-mlogloss:0.12469
[15]	train-mlogloss:0.15424	eval-mlogloss:0.11001
[16]	train-mlogloss:0.14399	eval-mlogloss:0.09882
[17]	train-mlogloss:0.13505	eval-mlogloss:0.09002
[18]	train-mlogloss:0.12594	eval-mlogloss:0.08295
[19]	train-mlogloss:0.11935	eval-mlogloss:0.07807
[20]	train

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning:

`optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.



[52]	train-mlogloss:0.07624	eval-mlogloss:0.03458
[53]	train-mlogloss:0.07625	eval-mlogloss:0.03480
[54]	train-mlogloss:0.07585	eval-mlogloss:0.03493
[55]	train-mlogloss:0.07509	eval-mlogloss:0.03393
[56]	train-mlogloss:0.07450	eval-mlogloss:0.03326
[57]	train-mlogloss:0.07402	eval-mlogloss:0.03236
[58]	train-mlogloss:0.07415	eval-mlogloss:0.03261
[59]	train-mlogloss:0.07378	eval-mlogloss:0.03258
[60]	train-mlogloss:0.07356	eval-mlogloss:0.03260
[61]	train-mlogloss:0.07370	eval-mlogloss:0.03277
[62]	train-mlogloss:0.07342	eval-mlogloss:0.03271
[63]	train-mlogloss:0.07298	eval-mlogloss:0.03202
[64]	train-mlogloss:0.07226	eval-mlogloss:0.03210
[65]	train-mlogloss:0.07236	eval-mlogloss:0.03194
[66]	train-mlogloss:0.07218	eval-mlogloss:0.03171
[67]	train-mlogloss:0.07180	eval-mlogloss:0.03126
[68]	train-mlogloss:0.07130	eval-mlogloss:0.03228
[69]	train-mlogloss:0.07097	eval-mlogloss:0.03137
[70]	train-mlogloss:0.07098	eval-mlogloss:0.03058
[71]	train-mlogloss:0.07094	eval-mlogloss:0.03023


...

[0]	train-mlogloss:1.07974	eval-mlogloss:1.08209
[1]	train-mlogloss:1.05398	eval-mlogloss:1.05518
[2]	train-mlogloss:1.02911	eval-mlogloss:1.02948
[3]	train-mlogloss:1.00468	eval-mlogloss:1.00440


[I 2026-06-28 08:49:23,232] Trial 48 pruned. Trial was pruned at iteration 4.


[0]	train-mlogloss:1.06688	eval-mlogloss:1.06739


[I 2026-06-28 08:49:23,272] Trial 49 pruned. Trial was pruned at iteration 1.


Best trial: {'lambda': 0.0003434360054112741, 'alpha': 1.557035917472026e-06, 'eta': 0.1568178961264044, 'gamma': 0.0025425644579019253, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.6677265794176557, 'colsample_bytree': 0.541111550069141}
Best accuracy: 1.0


In [37]:
! pip install optuna-integration[xgboost]

In [38]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()